# 09 — LSTM-CNN v2: Anti-Overfitting + Balanced Class Learning

**Problem diagnosed from 08c:**
- Train F1 >> Val/Test F1 → severe overfitting
- Minority classes (OTM10_30, ATM_90) poorly predicted despite resampling
- Weighted CrossEntropy alone insufficient for hard minority sequences

**Improvements in this notebook:**

| Area | 08c | v2 (this notebook) |
|------|-----|--------------------|
| Loss | Weighted CrossEntropy | **Focal Loss** (γ=2) — penalises easy examples, forces learning on hard ones |
| Augmentation | None | **Gaussian noise + feature masking + time shift** applied on-the-fly |
| Batch sampling | Random | **WeightedRandomSampler** — every batch sees equal class representation |
| Label smoothing | None | **ε=0.1** — prevents overconfident predictions |
| LR schedule | Constant | **Cosine Annealing with Warm Restarts** |
| Regularisation | Dropout 0.5 | Dropout + **SpatialDropout1d** on CNN + **weight decay sweep** |
| Resampling | Fixed targets | **Exact uniform resampling** to max-class count |
| Optuna trials | 7 | **20** (wider search: focal γ, mixup α, LR, weight decay) |
| Checkpointing | End only | **Every epoch** — spot-instance safe, resumable |
| Tracking | None | **MLflow** — params, metrics, confusion matrix, model artifact |


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import mlflow
import mlflow.pytorch

print('PyTorch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', device)

PyTorch: 2.11.0
Device : cpu


## 2. Configuration

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH  = Path('../data/clean/daily_stock_optimal_bucket_modeling_with_fred.parquet')
MODEL_DIR  = Path('../saved_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH  = MODEL_DIR / 'lstm_cnn_v2_checkpoint.pth'   # per-epoch checkpoint (spot safe)
BEST_PATH  = MODEL_DIR / 'lstm_cnn_v2_best.pth'         # best val F1 model

# ── Sequence config ────────────────────────────────────────────────────────
SEQ_LEN        = 50
N_TOP_FEATURES = 35
RANDOM_STATE   = 42

# ── Training ───────────────────────────────────────────────────────────────
N_EPOCHS       = 60      # more epochs; cosine LR + early stop will handle it
PATIENCE       = 12      # early stopping patience
OPTUNA_TRIALS  = 20
OPTUNA_EPOCHS  = 20      # epochs per Optuna trial (fast search)

# ── MLflow ─────────────────────────────────────────────────────────────────
MLFLOW_URI     = 'http://localhost:5001'   # local run
EXPERIMENT     = 'covered-call-lstm-cnn-v2'

print(f'Data path exists: {DATA_PATH.exists()}')
print(f'SEQ_LEN={SEQ_LEN}, N_TOP_FEATURES={N_TOP_FEATURES}')
print(f'Saving checkpoints to: {MODEL_DIR}')


Data path exists: True
SEQ_LEN=50, N_TOP_FEATURES=35
Saving checkpoints to: ../saved_models


## 3. Data Loading & Bucket Remapping

In [3]:
df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

print('Raw shape      :', df.shape)
print('Date range     :', df['date'].min().date(), '→', df['date'].max().date())
print('Symbols        :', df['symbol'].nunique())
print('\nClass distribution (raw):')
print(df['optimal_bucket'].value_counts())

Raw shape      : (41266, 88)
Date range     : 2006-01-03 → 2024-12-31
Symbols        : 10

Class distribution (raw):
optimal_bucket
ATM_30      12925
ATM_60       9494
ATM_90       6683
OTM10_30     4590
OTM5_30      4225
OTM10_90     1740
OTM10_60      781
OTM5_90       525
OTM5_60       303
Name: count, dtype: int64


In [4]:
# Merge longer-dated OTM buckets
bucket_map = {
    'OTM10_60': 'OTM10_60_90',
    'OTM10_90': 'OTM10_60_90',
    'OTM5_60' : 'OTM5_60_90',
    'OTM5_90' : 'OTM5_60_90',
}
df['optimal_bucket'] = df['optimal_bucket'].replace(bucket_map)

target_encoder = LabelEncoder()
df['target']   = target_encoder.fit_transform(df['optimal_bucket'])

num_classes = len(target_encoder.classes_)
class_names = list(target_encoder.classes_)

print('Classes after merge:', class_names)
print('\nDistribution after merge:')
print(df['optimal_bucket'].value_counts())

Classes after merge: ['ATM_30', 'ATM_60', 'ATM_90', 'OTM10_30', 'OTM10_60_90', 'OTM5_30', 'OTM5_60_90']

Distribution after merge:
optimal_bucket
ATM_30         12925
ATM_60          9494
ATM_90          6683
OTM10_30        4590
OTM5_30         4225
OTM10_60_90     2521
OTM5_60_90       828
Name: count, dtype: int64


## 4. Temporal Train / Val / Test Split

In [5]:
train_df = df[df['date'] <  '2022-01-01'].copy()
val_df   = df[(df['date'] >= '2022-01-01') & (df['date'] < '2024-01-01')].copy()
test_df  = df[df['date'] >= '2024-01-01'].copy()

print('Train:', train_df.shape, '|', train_df['date'].min().date(), '→', train_df['date'].max().date())
print('Val  :', val_df.shape,   '|', val_df['date'].min().date(),   '→', val_df['date'].max().date())
print('Test :', test_df.shape,  '|', test_df['date'].min().date(),  '→', test_df['date'].max().date())

Train: (33736, 89) | 2006-01-03 → 2021-12-31
Val  : (5010, 89) | 2022-01-03 → 2023-12-29
Test : (2520, 89) | 2024-01-02 → 2024-12-31


## 5. Feature Selection (Top 35 via Random Forest)

In [6]:
exclude_cols = ['symbol', 'date', 'fiscalDateEnding', 'optimal_bucket', 'target']
all_feature_cols = [c for c in train_df.columns if c not in exclude_cols]

X_rf = train_df[all_feature_cols].select_dtypes(include=['number']).fillna(0)
y_rf = train_df['target']

print(f'Training RF on {X_rf.shape[1]} numeric features …')
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_rf, y_rf)

importance_df = pd.DataFrame({
    'feature'   : X_rf.columns,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

feature_cols = importance_df.head(N_TOP_FEATURES)['feature'].tolist()
print(f'Top {N_TOP_FEATURES} features selected.')
print(importance_df.head(10).to_string(index=False))

Training RF on 84 numeric features …
Top 35 features selected.
           feature  importance
          CPIAUCSL    0.059577
            ma_200    0.035904
      longTermDebt    0.035505
    dividendPayout    0.031684
      gross_margin    0.028011
             ma_50    0.026300
  totalLiabilities    0.025784
totalCurrentAssets    0.025128
             ma_20    0.021280
       rolling_max    0.019586


## 6. Standard Scaling

In [7]:
scaler = StandardScaler()

for split_df in [train_df, val_df, test_df]:
    for col in feature_cols:
        if col not in split_df.columns:
            split_df[col] = 0.0
    split_df[feature_cols] = split_df[feature_cols].fillna(
        split_df[feature_cols].median()
    )

scaler.fit(train_df[feature_cols])
train_df[feature_cols] = scaler.transform(train_df[feature_cols])
val_df[feature_cols]   = scaler.transform(val_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])
print('Scaling complete.')

Scaling complete.


## 7. Sequence Construction

In [8]:
def build_sequences(panel_df, feature_cols, target_col='target', seq_len=50):
    X_seq, y_seq = [], []
    for sym, grp in panel_df.groupby('symbol'):
        grp    = grp.sort_values('date').reset_index(drop=True)
        X_vals = grp[feature_cols].values.astype(np.float32)
        y_vals = grp[target_col].values
        if len(grp) < seq_len:
            continue
        for i in range(seq_len - 1, len(grp)):
            X_seq.append(X_vals[i - seq_len + 1: i + 1])
            y_seq.append(y_vals[i])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.int64)

X_train_seq, y_train_seq = build_sequences(train_df, feature_cols, seq_len=SEQ_LEN)
X_val_seq,   y_val_seq   = build_sequences(val_df,   feature_cols, seq_len=SEQ_LEN)
X_test_seq,  y_test_seq  = build_sequences(test_df,  feature_cols, seq_len=SEQ_LEN)

print(f'Train: {X_train_seq.shape}  Val: {X_val_seq.shape}  Test: {X_test_seq.shape}')
print('\nClass distribution in train sequences:')
unique, counts = np.unique(y_train_seq, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {class_names[u]:15s}: {c:6d}')

Train: (33246, 50, 35)  Val: (4520, 50, 35)  Test: (2030, 50, 35)

Class distribution in train sequences:
  ATM_30         :  11728
  ATM_60         :   8538
  ATM_90         :   4767
  OTM10_30       :   3637
  OTM10_60_90    :    422
  OTM5_30        :   3798
  OTM5_60_90     :    356


## 8. Class Distribution Analysis

**No oversampling applied** — random oversampling across symbols and time periods breaks temporal integrity:
- A sequence from AAPL 2018 duplicated next to MSFT 2021 = data leakage
- Rolling windows are autocorrelated; duplicating them inflates effective sample size artificially

**How we handle imbalance instead (no data fabrication):**
- `WeightedRandomSampler` — ensures every training batch sees equal class representation by sampling minority sequences more frequently
- `Focal Loss` — forces the model to focus gradient on hard/minority examples
- On-the-fly augmentation (noise, masking, time shift) applied **only to minority classes** during training

In [9]:
# No oversampling — use original temporal sequences
# WeightedRandomSampler (in DataLoader) handles class balance at batch level
X_train_bal = X_train_seq
y_train_bal = y_train_seq

print('Training class distribution (original, no resampling):')
unique, counts = np.unique(y_train_bal, return_counts=True)
for u, c in zip(unique, counts):
    ratio = c / len(y_train_bal)
    print(f'  {class_names[u]:15s}: {c:6d}  ({ratio:.1%})')
print(f'Total train sequences: {len(y_train_bal):,}')

# Class imbalance ratio — drives WeightedRandomSampler and Focal Loss alpha
max_count = counts.max()
print(f'\nImbalance ratio (max/min): {max_count / counts.min():.1f}x')

Training class distribution (original, no resampling):
  ATM_30         :  11728  (35.3%)
  ATM_60         :   8538  (25.7%)
  ATM_90         :   4767  (14.3%)
  OTM10_30       :   3637  (10.9%)
  OTM10_60_90    :    422  (1.3%)
  OTM5_30        :   3798  (11.4%)
  OTM5_60_90     :    356  (1.1%)
Total train sequences: 33,246

Imbalance ratio (max/min): 32.9x


## 9. Dataset with Class-Aware On-the-Fly Augmentation

Augmentation is applied **only during training** and **heavier for minority classes**:
- **Gaussian noise** — σ scales with inverse class frequency (minority gets more noise diversity)
- **Feature masking** — randomly zeros ~5% of features per timestep
- **Time shift** (±2 steps) — small circular shift of the sequence
- `WeightedRandomSampler` — each batch is class-balanced without touching the data

In [10]:
class AugmentedDataset(Dataset):
    """
    Time-series dataset with class-aware on-the-fly augmentation.
    Minority classes receive heavier augmentation to increase diversity
    without duplicating or resampling sequences.
    """
    def __init__(self, X, y, augment=False,
                 base_noise=0.005, mask_prob=0.05, max_shift=2,
                 class_noise_multiplier=None):
        self.X                    = torch.tensor(X, dtype=torch.float32)
        self.y                    = torch.tensor(y, dtype=torch.long)
        self.augment              = augment
        self.base_noise           = base_noise
        self.mask_prob            = mask_prob
        self.max_shift            = max_shift
        # Per-class noise multiplier: minority classes get higher noise std
        self.class_noise_mult     = class_noise_multiplier  # dict: {class_idx: multiplier}

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x   = self.X[idx].clone()   # (seq_len, n_features)
        cls = self.y[idx].item()
        if self.augment:
            # Scale noise by class frequency — minority classes get more variation
            mult      = self.class_noise_mult.get(cls, 1.0) if self.class_noise_mult else 1.0
            noise_std = self.base_noise * mult
            # 1. Gaussian noise
            x = x + torch.randn_like(x) * noise_std
            # 2. Feature masking
            mask = torch.rand_like(x) > self.mask_prob
            x    = x * mask
            # 3. Small time shift (preserves temporal order within the shift)
            shift = torch.randint(-self.max_shift, self.max_shift + 1, (1,)).item()
            if shift != 0:
                x = torch.roll(x, shifts=shift, dims=0)
        return {'past_values': x, 'target': self.y[idx]}


# Compute per-class noise multiplier: inverse of normalised class frequency
unique, counts = np.unique(y_train_bal, return_counts=True)
freq           = counts / counts.sum()
inv_freq       = 1.0 / freq
inv_freq_norm  = inv_freq / inv_freq.min()   # minority class = highest multiplier
class_noise_multiplier = {int(cls): float(mult)
                           for cls, mult in zip(unique, inv_freq_norm)}
print('Per-class augmentation noise multiplier:')
for cls, mult in class_noise_multiplier.items():
    print(f'  {class_names[cls]:15s}: {mult:.2f}x')


def make_loader(X, y, batch_size, augment=False, balanced_sampling=False):
    dataset = AugmentedDataset(
        X, y, augment=augment,
        class_noise_multiplier=class_noise_multiplier if augment else None,
    )
    sampler = None
    if balanced_sampling:
        # WeightedRandomSampler: each batch sees equal class distribution
        class_counts = np.bincount(y)
        weights      = 1.0 / class_counts[y]
        sampler      = WeightedRandomSampler(
            weights=torch.tensor(weights, dtype=torch.float),
            num_samples=len(y),
            replacement=True,
        )
    return DataLoader(
        dataset, batch_size=batch_size,
        shuffle=(sampler is None and augment),
        sampler=sampler,
        num_workers=0, pin_memory=False,
    )

print('\nDataset and loader defined.')

Per-class augmentation noise multiplier:
  ATM_30         : 1.00x
  ATM_60         : 1.37x
  ATM_90         : 2.46x
  OTM10_30       : 3.22x
  OTM10_60_90    : 27.79x
  OTM5_30        : 3.09x
  OTM5_60_90     : 32.94x

Dataset and loader defined.


## 10. Focal Loss

**Why Focal Loss over Weighted CrossEntropy:**
- Weighted CE treats all minority samples equally regardless of difficulty
- Focal Loss down-weights easy examples (already well-classified) and focuses gradient on hard/misclassified ones
- `γ=2` is standard; combined with `alpha` (class weights) gives the best of both worlds

In [11]:
class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss with optional per-class alpha weights and label smoothing.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Parameters
    ----------
    gamma        : focusing parameter (0 = standard CE, 2 = standard focal)
    alpha        : per-class weight tensor (same as class_weight in sklearn)
    label_smooth : label smoothing epsilon (0 = off)
    """
    def __init__(self, gamma=2.0, alpha=None, label_smooth=0.1, num_classes=7):
        super().__init__()
        self.gamma        = gamma
        self.label_smooth = label_smooth
        self.num_classes  = num_classes
        if alpha is not None:
            self.register_buffer('alpha', alpha.float())
        else:
            self.alpha = None

    def forward(self, logits, targets):
        # Label smoothing
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits)
            smooth_targets.fill_(self.label_smooth / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smooth)

        log_probs = F.log_softmax(logits, dim=-1)
        probs     = log_probs.exp()

        # Focal weight: (1 - p_t)^gamma
        p_t       = (probs * F.one_hot(targets, self.num_classes).float()).sum(dim=-1)
        focal_w   = (1.0 - p_t) ** self.gamma

        # Cross-entropy with smooth targets
        ce_loss   = -(smooth_targets * log_probs).sum(dim=-1)

        # Alpha weighting
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss    = alpha_t * focal_w * ce_loss
        else:
            loss    = focal_w * ce_loss

        return loss.mean()


# Compute class weights from balanced training set
classes_arr   = np.sort(np.unique(y_train_bal))
class_weights = compute_class_weight('balanced', classes=classes_arr, y=y_train_bal)
alpha_tensor  = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class alpha weights:', {class_names[k]: round(float(v), 3)
                                for k, v in enumerate(class_weights)})

Class alpha weights: {'ATM_30': 0.405, 'ATM_60': 0.556, 'ATM_90': 0.996, 'OTM10_30': 1.306, 'OTM10_60_90': 11.255, 'OTM5_30': 1.251, 'OTM5_60_90': 13.341}


## 11. LSTM-CNN v2 Architecture

**Additions vs 08c:**
- `SpatialDropout1d` on CNN features (drops entire feature channels, not individual values)
- Residual connection in CNN branch
- `LayerNorm` after LSTM (stabilises training)

In [12]:
class SpatialDropout1d(nn.Module):
    """Drops entire channels (feature maps) rather than individual values."""
    def __init__(self, p):
        super().__init__()
        self.p = p

    def forward(self, x):   # x: (batch, channels, length)
        if not self.training or self.p == 0:
            return x
        mask = torch.bernoulli(torch.ones(x.shape[0], x.shape[1], 1, device=x.device) * (1 - self.p))
        return x * mask / (1 - self.p)


class MultiScaleCNNv2(nn.Module):
    """Multi-scale CNN with SpatialDropout and residual projection."""
    def __init__(self, n_features, cnn_out_channels, dropout):
        super().__init__()
        self.spatial_drop = SpatialDropout1d(dropout * 0.5)
        self.branches = nn.ModuleList([
            self._make_branch(n_features, cnn_out_channels, k, dropout)
            for k in [3, 5, 7]
        ])

    @staticmethod
    def _make_branch(in_ch, out_ch, kernel, dropout):
        pad = kernel // 2
        return nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=pad),
            nn.BatchNorm1d(out_ch), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size=kernel, padding=pad),
            nn.BatchNorm1d(out_ch), nn.GELU(), nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):   # x: (batch, n_features, seq_len)
        x = self.spatial_drop(x)
        return torch.cat([b(x).squeeze(-1) for b in self.branches], dim=-1)


class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim, attn_dim=64):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, lstm_out):
        w       = torch.softmax(self.v(torch.tanh(self.W(lstm_out))), dim=1)
        context = (w * lstm_out).sum(dim=1)
        return context, w.squeeze(-1)


class LSTMCNNv2(nn.Module):
    def __init__(self, n_features, seq_len, num_classes,
                 cnn_out_channels=64, lstm_hidden=128,
                 lstm_layers=2, attn_dim=64, dropout=0.3):
        super().__init__()
        self.cnn  = MultiScaleCNNv2(n_features, cnn_out_channels, dropout)
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True, bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.lstm_norm = nn.LayerNorm(lstm_hidden * 2)   # new: stabilise LSTM output
        self.attention = TemporalAttention(lstm_hidden * 2, attn_dim)
        fusion_dim     = cnn_out_channels * 3 + lstm_hidden * 2
        self.head      = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim // 2), nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(fusion_dim // 2, num_classes),
        )

    def forward(self, past_values):   # (batch, seq_len, n_features)
        x_cnn          = self.cnn(past_values.transpose(1, 2))
        lstm_out, _    = self.lstm(past_values)
        lstm_out       = self.lstm_norm(lstm_out)
        x_attn, _      = self.attention(lstm_out)
        return self.head(torch.cat([x_cnn, x_attn], dim=-1))

print('LSTMCNNv2 architecture defined.')

LSTMCNNv2 architecture defined.


## 12. Training & Evaluation Helpers

**New vs 08c:** Cosine Annealing LR schedule + per-epoch checkpointing for spot-instance safety.

In [13]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, all_preds, all_targets = 0.0, [], []
    for batch in loader:
        x = batch['past_values'].to(device)
        y = batch['target'].to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_targets.extend(y.cpu().numpy())
    n    = len(loader.dataset)
    f1   = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return total_loss / n, f1


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_targets, all_probs = 0.0, [], [], []
    for batch in loader:
        x      = batch['past_values'].to(device)
        y      = batch['target'].to(device)
        logits = model(x)
        loss   = criterion(logits, y)
        probs  = torch.softmax(logits, dim=-1)
        total_loss += loss.item() * len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_targets.extend(y.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
    n         = len(loader.dataset)
    f1        = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    probs_arr = np.vstack(all_probs)
    return total_loss / n, f1, np.array(all_targets), np.array(all_preds), probs_arr


def save_checkpoint(model, optimizer, scheduler, epoch, best_val_f1, path):
    torch.save({
        'epoch'       : epoch,
        'model_state' : model.state_dict(),
        'optim_state' : optimizer.state_dict(),
        'sched_state' : scheduler.state_dict(),
        'best_val_f1' : best_val_f1,
    }, path)


def load_checkpoint(model, optimizer, scheduler, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optim_state'])
    scheduler.load_state_dict(ckpt['sched_state'])
    return ckpt['epoch'], ckpt['best_val_f1']

print('Training helpers defined.')

Training helpers defined.


## 13. Optuna Hyperparameter Search (20 Trials)

**New search dimensions vs 08c:**
- `focal_gamma`: how hard to focus on difficult examples (0.5–3.0)
- `label_smooth`: label smoothing epsilon (0.0–0.2)
- Wider `lr` range (1e-4 to 3e-3)
- `weight_decay` range extended to 1e-1

In [ ]:
def objective(trial):
    cnn_out_channels = trial.suggest_categorical('cnn_out_channels', [32, 64, 128])
    lstm_hidden      = trial.suggest_categorical('lstm_hidden',      [64, 128, 256])
    lstm_layers      = trial.suggest_int('lstm_layers', 1, 3)
    attn_dim         = trial.suggest_categorical('attn_dim',         [32, 64, 128])
    dropout          = trial.suggest_float('dropout', 0.2, 0.5)
    lr               = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    weight_decay     = trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True)
    batch_size       = trial.suggest_categorical('batch_size', [64, 128, 256])
    focal_gamma      = trial.suggest_float('focal_gamma', 0.5, 3.0)
    label_smooth     = trial.suggest_float('label_smooth', 0.0, 0.2)

    model = LSTMCNNv2(
        n_features       = N_TOP_FEATURES,
        seq_len          = SEQ_LEN,
        num_classes      = num_classes,
        cnn_out_channels = cnn_out_channels,
        lstm_hidden      = lstm_hidden,
        lstm_layers      = lstm_layers,
        attn_dim         = attn_dim,
        dropout          = dropout,
    ).to(device)

    criterion = FocalLoss(
        gamma        = focal_gamma,
        alpha        = alpha_tensor,
        label_smooth = label_smooth,
        num_classes  = num_classes,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=OPTUNA_EPOCHS)

    train_loader = make_loader(X_train_bal, y_train_bal, batch_size,
                               augment=True, balanced_sampling=True)
    val_loader   = make_loader(X_val_seq,   y_val_seq,   128, augment=False)

    best_val_f1 = 0.0
    for epoch in range(OPTUNA_EPOCHS):
        train_epoch(model, train_loader, criterion, optimizer, device)
        scheduler.step()
        _, val_f1, _, _, _ = eval_epoch(model, val_loader, criterion, device)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_f1


print(f'Running {OPTUNA_TRIALS} Optuna trials …')
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
study   = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_params = study.best_trial.params
print(f'\nBest Val Macro-F1: {study.best_value:.4f}')
print('Best params:', best_params)

Running 20 Optuna trials …


  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
# Optuna trial visualisation
trial_values = [t.value for t in study.trials if t.value is not None]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(trial_values) + 1), trial_values, marker='o', color='steelblue')
ax.axhline(max(trial_values), color='red', linestyle='--',
           label=f'Best = {max(trial_values):.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('Val Macro-F1')
ax.set_title('Optuna — 20 Trials Macro-F1')
ax.legend()
plt.tight_layout()
plt.show()

## 14. Final Training with Best Hyperparameters

**Key differences from 08c:**
- Cosine Annealing LR with Warm Restarts (T_0=15, T_mult=2)
- Per-epoch checkpoint saves → spot interruption safe
- WeightedRandomSampler + augmentation active throughout
- MLflow logs every epoch

In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)

p = best_params

train_loader = make_loader(X_train_bal, y_train_bal, p['batch_size'],
                           augment=True, balanced_sampling=True)
val_loader   = make_loader(X_val_seq,  y_val_seq,  128, augment=False)
test_loader  = make_loader(X_test_seq, y_test_seq, 128, augment=False)

best_model = LSTMCNNv2(
    n_features       = N_TOP_FEATURES,
    seq_len          = SEQ_LEN,
    num_classes      = num_classes,
    cnn_out_channels = p['cnn_out_channels'],
    lstm_hidden      = p['lstm_hidden'],
    lstm_layers      = p['lstm_layers'],
    attn_dim         = p['attn_dim'],
    dropout          = p['dropout'],
).to(device)

criterion = FocalLoss(
    gamma        = p['focal_gamma'],
    alpha        = alpha_tensor,
    label_smooth = p['label_smooth'],
    num_classes  = num_classes,
)
optimizer = torch.optim.AdamW(best_model.parameters(),
                              lr=p['lr'], weight_decay=p['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=15, T_mult=2, eta_min=1e-6
)

history    = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
best_val_f1 = 0.0
no_improve  = 0
start_epoch = 0

# ── Resume from checkpoint if it exists (spot-safe) ───────────────────────
if CKPT_PATH.exists():
    print(f'Resuming from checkpoint: {CKPT_PATH}')
    start_epoch, best_val_f1 = load_checkpoint(
        best_model, optimizer, scheduler, CKPT_PATH, device
    )
    start_epoch += 1
    print(f'  Resuming from epoch {start_epoch}, best val F1 so far: {best_val_f1:.4f}')

with mlflow.start_run(run_name='lstm_cnn_v2_final') as run:
    mlflow.log_params({
        **p,
        'seq_len'     : SEQ_LEN,
        'n_features'  : N_TOP_FEATURES,
        'num_classes' : num_classes,
        'n_epochs'    : N_EPOCHS,
        'patience'    : PATIENCE,
    })

    for epoch in range(start_epoch, N_EPOCHS):
        tr_loss, tr_f1 = train_epoch(best_model, train_loader, criterion, optimizer, device)
        vl_loss, vl_f1, _, _, _ = eval_epoch(best_model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_f1'].append(tr_f1)
        history['val_f1'].append(vl_f1)

        mlflow.log_metrics({
            'train_loss': tr_loss, 'val_loss': vl_loss,
            'train_f1'  : tr_f1,  'val_f1'  : vl_f1,
            'lr'        : optimizer.param_groups[0]['lr'],
        }, step=epoch)

        # ── Per-epoch checkpoint (spot safe) ─────────────────────────────
        save_checkpoint(best_model, optimizer, scheduler, epoch, best_val_f1, CKPT_PATH)

        # ── Save best model ───────────────────────────────────────────────
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            no_improve  = 0
            torch.save(best_model.state_dict(), BEST_PATH)
        else:
            no_improve += 1

        print(f'Ep {epoch+1:3d}/{N_EPOCHS} | '
              f'TrLoss={tr_loss:.4f} TrF1={tr_f1:.4f} | '
              f'VlLoss={vl_loss:.4f} VlF1={vl_f1:.4f} | '
              f'Best={best_val_f1:.4f} | NoImp={no_improve}')

        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}.')
            break

    print(f'\nTraining complete. Best val Macro-F1: {best_val_f1:.4f}')
    print(f'MLflow Run ID: {run.info.run_id}')

## 15. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes  = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_ran, history['train_loss'], label='Train')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val')
axes[0].set_title('Focal Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_ran, history['train_f1'], label='Train')
axes[1].plot(epochs_ran, history['val_f1'],   label='Val')
axes[1].set_title('Macro-F1')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Annotate train-val gap at last epoch
last = -1
gap  = history['train_f1'][last] - history['val_f1'][last]
fig.suptitle(f'LSTM-CNN v2 | Final train-val F1 gap: {gap:+.4f}', fontsize=12)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'lstm_cnn_v2_training_curves.png', dpi=120)
plt.show()

## 16. Load Best Model & Evaluate on Test

In [ ]:
# Load best saved weights
best_model.load_state_dict(torch.load(BEST_PATH, map_location=device))
best_model.eval()

_, val_f1,  y_val_true,  y_val_pred,  y_val_probs  = eval_epoch(
    best_model, val_loader,  criterion, device)
_, test_f1, y_tst_true,  y_tst_pred,  y_tst_probs  = eval_epoch(
    best_model, test_loader, criterion, device)

def print_metrics(y_true, y_pred, split):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bac = balanced_accuracy_score(y_true, y_pred)
    print(f'\n── {split} ──')
    print(f'  Accuracy          : {acc:.4f}')
    print(f'  Macro-F1          : {f1:.4f}')
    print(f'  Balanced Accuracy : {bac:.4f}')
    print(f'  Per-class F1:')
    per = f1_score(y_true, y_pred, average=None, zero_division=0)
    for cls, pf1 in zip(class_names, per):
        print(f'    {cls:15s}: {pf1:.4f}')

print_metrics(y_val_true,  y_val_pred,  'Validation (argmax)')
print_metrics(y_tst_true,  y_tst_pred,  'Test (argmax)')

## 17. Threshold Tuning

In [ ]:
def predict_with_thresholds(probs, thresholds):
    return (probs / np.array(thresholds).reshape(1, -1)).argmax(axis=1)


def tune_thresholds(y_true, probs, n_classes, rounds=3):
    thresholds  = np.ones(n_classes)
    search_grid = np.arange(0.2, 1.05, 0.05)
    for _ in range(rounds):
        for c in range(n_classes):
            best_t, best_f1 = thresholds[c], 0.0
            for t in search_grid:
                thresholds[c] = t
                preds = predict_with_thresholds(probs, thresholds)
                f1    = f1_score(y_true, preds, average='macro', zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, t
            thresholds[c] = best_t
    return thresholds


tuned_thresholds = tune_thresholds(y_val_true, y_val_probs, num_classes)
print('Tuned thresholds:', {class_names[i]: round(t, 2)
                            for i, t in enumerate(tuned_thresholds)})

y_val_tuned = predict_with_thresholds(y_val_probs, tuned_thresholds)
y_tst_tuned = predict_with_thresholds(y_tst_probs, tuned_thresholds)

print_metrics(y_val_tuned, y_val_tuned, 'Validation (threshold-tuned)')
print_metrics(y_tst_true,  y_tst_tuned, 'Test (threshold-tuned)')

## 18. Confusion Matrix

In [ ]:
def plot_cm(y_true, y_pred, labels, title, save_path=None):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()

plot_cm(y_val_true, y_val_tuned, class_names,
        'Confusion Matrix — Validation (threshold-tuned)',
        save_path=MODEL_DIR / 'lstm_cnn_v2_cm_val.png')

plot_cm(y_tst_true, y_tst_tuned, class_names,
        'Confusion Matrix — Test (threshold-tuned)',
        save_path=MODEL_DIR / 'lstm_cnn_v2_cm_test.png')

## 19. ROC-AUC & Precision-Recall Curves

In [ ]:
def plot_roc(y_true, probs, n_classes, names, title):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(9, 7))
    for i, name in enumerate(names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--')
    ax.set(xlabel='FPR', ylabel='TPR', title=title)
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

def plot_pr(y_true, probs, n_classes, names, title):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(9, 7))
    for i, name in enumerate(names):
        p, r, _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        ap      = average_precision_score(y_bin[:, i], probs[:, i])
        ax.plot(r, p, label=f'{name} (AP={ap:.2f})')
    ax.set(xlabel='Recall', ylabel='Precision', title=title)
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

plot_roc(y_tst_true, y_tst_probs, num_classes, class_names, 'ROC-AUC — Test')
plot_pr(y_tst_true,  y_tst_probs, num_classes, class_names, 'Precision-Recall — Test')

## 20. Metrics Summary

In [ ]:
rows = []
for split, y_true, y_pred_base, y_pred_tuned in [
    ('Validation', y_val_true, y_val_pred, y_val_tuned),
    ('Test',       y_tst_true, y_tst_pred, y_tst_tuned),
]:
    for method, y_pred in [('Argmax', y_pred_base), ('Threshold-Tuned', y_pred_tuned)]:
        rows.append({
            'Split'   : split,
            'Method'  : method,
            'Accuracy': round(accuracy_score(y_true, y_pred), 4),
            'Macro-F1': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
            'Bal-Acc' : round(balanced_accuracy_score(y_true, y_pred), 4),
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

## 21. Save Final Model + Log to MLflow

In [ ]:
FINAL_PATH = MODEL_DIR / 'lstm_cnn_v2_final.pth'

torch.save({
    'model_state_dict' : best_model.state_dict(),
    'best_params'      : best_params,
    'feature_cols'     : feature_cols,
    'seq_len'          : SEQ_LEN,
    'num_classes'      : num_classes,
    'target_classes'   : class_names,
    'tuned_thresholds' : tuned_thresholds.tolist(),
    'focal_gamma'      : best_params['focal_gamma'],
    'label_smooth'     : best_params['label_smooth'],
}, FINAL_PATH)

print(f'Model saved to: {FINAL_PATH}')

# ── Log final metrics + model artifact to MLflow ──────────────────────────
with mlflow.start_run(run_name='lstm_cnn_v2_final_eval') as run:
    test_f1_tuned = f1_score(y_tst_true, y_tst_tuned, average='macro', zero_division=0)
    mlflow.log_metrics({
        'test_macro_f1_argmax'  : float(test_f1),
        'test_macro_f1_tuned'   : float(test_f1_tuned),
        'val_macro_f1_argmax'   : float(val_f1),
        'train_val_f1_gap'      : float(history['train_f1'][-1] - history['val_f1'][-1]),
    })
    for i, (cls, thr) in enumerate(zip(class_names, tuned_thresholds)):
        mlflow.log_metric(f'threshold_{cls}', float(thr))
    mlflow.log_artifact(str(FINAL_PATH), artifact_path='checkpoint')
    mlflow.log_artifact(str(MODEL_DIR / 'lstm_cnn_v2_training_curves.png'))
    mlflow.log_artifact(str(MODEL_DIR / 'lstm_cnn_v2_cm_test.png'))
    mlflow.pytorch.log_model(
        best_model,
        artifact_path='model',
        registered_model_name='CoveredCallLSTMCNN_v2',
    )
    print(f'MLflow run: {run.info.run_id}')

---
## Summary

| Step | Detail |
|------|--------|
| Data | `daily_stock_optimal_bucket_modeling_with_fred.parquet` |
| Target classes | 7 covered-call buckets (ATM_30/60/90, OTM5_30/60_90, OTM10_30/60_90) |
| Features | Top 35 (Random Forest importance) |
| Sequence length | 50 trading days |
| Train / Val / Test | < 2022 / 2022–2023 / ≥ 2024 |
| Imbalance fix | Uniform oversampling to max-class count + Gaussian noise on duplicates |
| Loss | Focal Loss (γ tuned by Optuna) + label smoothing |
| Augmentation | Gaussian noise + feature masking + time shift (train only) |
| Batch sampling | WeightedRandomSampler (class-balanced batches) |
| LR schedule | Cosine Annealing Warm Restarts |
| Architecture | LSTM-CNN v2: SpatialDropout1d, GELU, LayerNorm on LSTM output |
| Optuna trials | 20 (focal_gamma, label_smooth added to search) |
| Checkpointing | Per-epoch — resumable after spot interruption |
| Tracking | MLflow: params, per-epoch metrics, confusion matrix, registered model |
| Saved model | `saved_models/lstm_cnn_v2_final.pth` |